In [2]:
import json
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

In [3]:
rows = []
for path in sorted(RAW_DIR.glob("*.json")):
    page = json.loads(path.read_text(encoding="utf-8-sig"))
    for field in page["fields"]:
        rows.append({
            **field,
            "site": page["site"],
            "version": page["extractorVersion"],
            "field_id": f"{path.stem}#{field['order']}",
        })

df = pd.DataFrame(rows)
df.shape

(366, 18)

In [4]:
df = df[df["version"].str.startswith("1.1")]
df["version"].value_counts()

version
1.1    366
Name: count, dtype: int64

In [5]:
is_twin = (
    ~df["visible"]
    & (df["id"] == "")
    & (df["name"] == "")
    & (df["type"] != "file")
)
print("Dropping", is_twin.sum(), "hidden value-holders")
df = df[~is_twin]
df.shape

Dropping 27 hidden value-holders


(339, 18)

In [6]:
columns = [
    "field_id", "site", "section", "group", "label", "option",
    "type", "tag", "name", "placeholder", "autocomplete",
    "automationId", "fieldPath", "required", "visible",
]
table = df[columns].copy()
table["category"] = ""
table.head(10)

,field_id,site,section,group,label,option,type,tag,name,placeholder,autocomplete,automationId,fieldPath,required,visible,category
0,apply.careers.microsoft.com_20261111-231816#0,apply.careers.microsoft.com,Application location(s),Application location(s),Where would you like to apply for Software Eng...,,text,INPUT,,,,,,True,True,
1,apply.careers.microsoft.com_20261111-231816#1,apply.careers.microsoft.com,Resume,Resume,Upload your resume (U.S. applicants only: in c...,,text,INPUT,,Select,,,,True,True,
2,apply.careers.microsoft.com_20261111-231816#2,apply.careers.microsoft.com,Resume,Resume,Resume,,file,INPUT,,,,,,False,False,
3,apply.careers.microsoft.com_20261111-231816#3,apply.careers.microsoft.com,Contact Information,Contact Information,Preferred First Name,,text,INPUT,,,given-name,,,True,True,
4,apply.careers.microsoft.com_20261111-231816#4,apply.careers.microsoft.com,Contact Information,Contact Information,Preferred Last Name,,text,INPUT,,,family-name,,,True,True,
5,apply.careers.microsoft.com_20261111-231816#5,apply.careers.microsoft.com,Contact Information,Contact Information,Is your legal name the same as your preferred ...,,text,INPUT,,Select,,,,True,True,
6,apply.careers.microsoft.com_20261111-231816#6,apply.careers.microsoft.com,Contact Information,Contact Information,Email,,text,INPUT,,,email,,,True,True,
7,apply.careers.microsoft.com_20261111-231816#7,apply.careers.microsoft.com,Contact Information,Phone,Country code,,text,INPUT,,Select country code,,,,True,True,
8,apply.careers.microsoft.com_20261111-231816#8,apply.careers.microsoft.com,Contact Information,Phone,Phone number,,tel,INPUT,,Phone number,,,,True,True,
9,apply.careers.microsoft.com_20261111-231816#9,apply.careers.microsoft.com,Contact Information,Contact Information,Alternate Phone Number,,text,INPUT,,,one-time-code,,,False,True,


In [7]:
PROCESSED_DIR.mkdir(exist_ok=True)
out = PROCESSED_DIR / "labeled_v1.csv"

if out.exists():
    print("labeled_v1.csv already exists, not overwriting your labels.")
else:
    table.to_csv(out, index=False, encoding="utf-8-sig")
    print("Saved", len(table), "rows to", out)

Saved 339 rows to ..\data\processed\labeled_v1.csv
